# LangPick — 言語指示で動く、視覚ベースの pick & place

**Physical AI 応用編1 / 最終課題**

指示文を変えるだけで、ロボットが掴む物と置き先が変わる。コードは変えない。

```
「赤いブロックを紫のトレイに入れて」
   │
   ├─ ① 指示解析    : 対象と置き先のスロットに分解
   ├─ ② 接地        : OWLv2（オープン語彙物体検出）が語を画素に結びつける
   ├─ ③ 3D 化       : RGB-D と内部パラメータから把持位置を逆投影で推定
   ├─ ④ 動作        : 減衰最小二乗 IK で把持・搬送・解放
   └─ ⑤ 自己検証    : 実行後の画像を再び VLM に通して成否を判定
```

**シミュレータ内部の物体座標は制御に一切使っていない**（評価のためだけに最後に参照する）。

---
### 実行手順
1. メニューの **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選ぶ（CPU でも動くが 10 倍遅い）
2. 上から順に全セルを実行する
3. 最後のセルで `PAI最終課題_<アカウント名>.mp4` がダウンロードされる

## 0. 設定

ここだけ書き換えれば動く。

In [ ]:
# あなたの omnicampus アカウント名（動画タイトルとファイル名に入る）
ACCOUNT = "your_omnicampus_id"

# このノートブックが入っているリポジトリ
REPO_URL = "https://github.com/<your-github-account>/langpick.git"

# 動画で実行する指示（文字列を変えるだけで対象が変わる）
INSTRUCTIONS = [
    "赤いブロックを紫のトレイに入れて",
    "青い缶を黄色のトレイに入れて",
    "赤い缶を紫のトレイに入れて",
]

## 1. 環境構築（約 2 分）

In [ ]:
import os, subprocess, sys

# MuJoCo をヘッドレスで描画するための設定。mujoco を import する前に必要
os.environ["MUJOCO_GL"] = "egl"

!pip -q install "mujoco>=3.2" imageio imageio-ffmpeg "transformers>=4.35" 2>&1 | tail -2
# 動画に日本語を焼き込むためのフォント
!apt-get -qq install -y fonts-noto-cjk > /dev/null 2>&1

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし（CPU で実行します）")

In [ ]:
import pathlib

REPO_DIR = pathlib.Path("/content/langpick")
if not REPO_DIR.exists():
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
sys.path.insert(0, str(REPO_DIR / "src"))
!ls src/langpick

## 2. シーンを作る

Panda は `mujoco_menagerie` から sparse checkout で取得する（Panda だけ・約 44MB）。
卓上には **赤いブロック / 赤い缶 / 緑のブロック / 青い缶** と、紫・黄のトレイを置く。
赤が 2 つあるのは意図的で、色だけを見る手法では区別できない配置にしてある。

In [ ]:
import matplotlib.pyplot as plt
from langpick import scene as S
from langpick.camera import Camera

model, data = S.make_model("assets", offwidth=960, offheight=540)
cam = Camera(model, 540, 960)
view = cam.capture(data, "percept")

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].imshow(view.rgb);                      ax[0].set_title("percept camera (RGB)")
im = ax[1].imshow(view.depth, cmap="magma"); ax[1].set_title("depth [m]")
plt.colorbar(im, ax=ax[1], fraction=0.03)
ax[2].imshow(cam.render_rgb(data, "cine"));  ax[2].set_title("cine camera")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 3. 逆投影の精度を確かめる

画像上の点を、深度とカメラ内部パラメータだけでワールド座標に戻せているか。
ここが合っていないと、この後の把持は全部外れる。

In [ ]:
import numpy as np
from langpick.experiment import project
from langpick.grasp import estimate_grasp

print(f"{'物体':<12}{'真値 (x, y)':<22}{'推定 (x, y)':<22}{'誤差':>8}")
for obj in S.OBJECTS:
    p = S.body_pos(model, data, obj.name)
    half = (obj.size[0], obj.size[0], obj.size[2] if obj.kind == "box" else obj.size[1])
    pts = np.array([project(view, p + np.array(half) * [sx, sy, sz])
                    for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)])
    box = (pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max())
    g = estimate_grasp(view, box)
    err = np.linalg.norm(g.position[:2] - p[:2]) * 1000
    print(f"{obj.name:<12}{str(np.round(p[:2], 4)):<22}"
          f"{str(np.round(g.position[:2], 4)):<22}{err:6.1f} mm")

## 4. 検出器を読み込む

OWLv2 は **学習済みモデルの推論のみ**（追加学習なし）。
テキストを与えるだけで未知クラスを検出できるので、指示文の語彙を増やしても再学習がいらない。

In [ ]:
import time
from langpick.perception import ColorBaseline, OwlV2Detector

t0 = time.time()
detector = OwlV2Detector()
print(f"読み込み {time.time() - t0:.1f} 秒 / device={detector.device}")

t0 = time.time()
dets = detector.detect(view.rgb, ["a red cube", "a red can", "a green cube",
                                  "a blue can", "a purple box", "a yellow box"], 0.08)
print(f"推論 {time.time() - t0:.2f} 秒")
for d in dets[:8]:
    print(f"  {d.query:<14} {d.score:.3f}  {[round(v) for v in d.box]}")

In [ ]:
from langpick.video import Banner, annotate

plt.figure(figsize=(12, 7))
plt.imshow(annotate(view.rgb, Banner("検出結果", "OWLv2 open-vocabulary detection",
                                     phase="PERCEPTION"), dets[:6], scale=1.5))
plt.axis("off"); plt.show()

## 5. 指示を 1 件だけ実行してみる

`INSTRUCTIONS[0]` の文字列を変えると、掴む物も置き先も変わる。コードは変えない。

In [ ]:
from langpick.episode import Env, evaluate_against_truth, run_episode

env = Env(detector)
result = run_episode(env, INSTRUCTIONS[0])
print(result.summary())
print("真値での確認:", evaluate_against_truth(result))

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].imshow(result.percept_rgb); ax[0].set_title("実行前（ここから把持位置を決めた）")
ax[1].imshow(result.after_rgb);   ax[1].set_title(f"実行後（自己検証: {result.verified}）")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 6. 対照実験 — VLM を使う必然性

色の閾値処理だけのベースラインと比べる。
赤いブロックと赤い缶は **同じ色** なので、色だけでは原理的に区別できない。

In [ ]:
from langpick import experiment

trials = experiment.run(env, {"OWLv2": detector, "色閾値": ColorBaseline()})
print(experiment.to_table(trials))

## 7. 提出用の動画を作る

タイトル・指示文・検出枠・フェーズ・判定をすべて焼き込んだ mp4 が 1 本出る。
**動画編集は不要**。GPU で約 5 分。

In [ ]:
sys.path.insert(0, str(REPO_DIR / "scripts"))
import make_video

# 読み込み済みの検出器を渡して、重みの二重ロードを避ける
summary = make_video.build(ACCOUNT, pathlib.Path("out"),
                           instructions=INSTRUCTIONS, detector=detector)
print(f"
長さ {summary['video_seconds']:.1f} 秒")

In [ ]:
from base64 import b64encode
from IPython.display import HTML

path = f"out/PAI最終課題_{ACCOUNT}.mp4"
src = "data:video/mp4;base64," + b64encode(open(path, "rb").read()).decode()
HTML(f'<video width=760 controls><source src="{src}" type="video/mp4"></video>')

In [ ]:
from google.colab import files

files.download(path)
files.download("out/results.json")   # レポートに貼る数値